# DEXTRA · 03 — Baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/03_baselines.ipynb)

The floor every later model has to clear, measured on the same splits and the same metrics the paper will report.

| Baseline | What it is | Why it is here |
|---|---|---|
| **Persistence** | tomorrow equals today | AQI is strongly autocorrelated; at one day ahead this is genuinely hard to beat |
| **Seasonal climatology** | each city's own average for that day of the year | answers "does the model know anything the calendar does not?" |
| **Gradient boosting** | a tree ensemble on the same 26 variates | if it comes close to the deep model, the architecture is not doing the work |

None of these is tuned. The point is a floor, not a rival.

### Runtime
**CPU.** The whole notebook fits in a couple of minutes and needs no GPU.

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

## Secrets
`HF_TOKEN` and `HF_DATASET_REPO`, as in the earlier notebooks.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow "scikit-learn>=1.4.0"

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from src import config
from src.data import assemble
from src.models import baselines, windows
from src.evaluation import metrics
from src.utils.hf_store import HFStore

store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
DATASET_PATH = "processed/city_daily_features.parquet"
daily = store.read_df(DATASET_PATH)
print(f"{len(daily):,} city-days, {daily.city.nunique()} cities, "
      f"{daily.date.min():%Y-%m-%d} to {daily.date.max():%Y-%m-%d}")

# The split labels are recomputed from the current config rather than trusted as
# saved. They are a pure function of the dates and the config, so this always
# agrees with notebook 02 when that has been re-run since the last change -- and
# says so when it has not, instead of scoring against a boundary that has moved.
saved = daily["split"].copy() if "split" in daily.columns else None
daily["split"] = assemble.assign_split(daily)
differing = int((saved != daily["split"]).sum()) if saved is not None else 0
if differing:
    print()
    print(f"! {differing:,} rows carry a different split than the saved dataset. "
          "Using the current config; re-run notebook 02 to bring the saved copy into line.")
print(f"forecasting {windows.TARGET} at +1, +2 and +3 days from a "
      f"{config.LOOKBACK_WINDOW}-day history")

## A · Build the supervised samples
One row per city, per day the forecast is made, per horizon. Three rules keep them honest:

- **Nothing after day *t* describes the input.** Every rolling window ends at *t*, every lag reaches backwards, and all shifts happen inside a city so one city's last days never become another's history.
- **A sample belongs to the split of the day it predicts**, not the day it is made from. A forecast issued on 31 December 2024 for 2 January 2025 is a test sample that happens to read validation history — exactly the position a real forecaster is in.
- **The calendar is known in advance.** Day of week, month, Diwali and the stubble season are read at the *target* date; everything measured is read at *t*.

The first of those is the one that would quietly inflate every result, so there is a test that wrecks all future values and asserts the past does not move.

In [ ]:
samples = windows.usable(windows.make_supervised(daily))
features = windows.feature_matrix(samples)
print(f"{len(samples):,} samples, {len(features)} features")
display(samples.split.value_counts().rename("samples").to_frame())

# What the model is allowed to read, grouped by where it comes from.
measured = [f for f in features if f in config.ALL_FEATURES]
history = [f for f in features if f.startswith(windows.TARGET)]
derived = [f for f in features if f not in measured and f not in history]
print(f"\n{len(measured)} measured at t, {len(history)} from the target's own history, "
      f"{len(derived)} derived: {derived}")

## B · Fit and score
Everything is fitted on the **training split only** — the ten training cities, 2019 to 2023. The unseen cities are never trained on and get no adaptation here: this is the zero-shot number the LoRA adapter has to improve on.

Climatology falls back to the pooled training seasonality for a city it never saw, because thirty days cannot describe a year.

In [ ]:
scored, models = baselines.run_baselines(samples, features)
MODELS = ["persistence", "climatology", "gbm"]
print("fitted on", f"{(samples.split == 'train').sum():,}", "training samples")

### B1 · Results
`rmse` and `mae` are in AQI points. `support` is how many days genuinely exceeded AQI 200, and `recall` the share of those the forecast also put above it — the number this project exists to improve.

In [ ]:
rows = []
for split in ["val", "test_seen", "test_unseen"]:
    part = scored[scored.split == split]
    for name in MODELS:
        row = {"split": split, "model": name}
        row.update(metrics.score(part, "y", name))
        rows.append(row)
table = pd.DataFrame(rows)[["split", "model", "n", "rmse", "mae",
                            "support", "recall", "precision", "f1"]]
display(table.round(3))

### B2 · The transfer gap
The same model, scored on cities it trained on and cities it has never seen — both across 2025, so the difference is about the cities and not about the year.

In [ ]:
gap = []
for name in MODELS:
    seen = metrics.score(scored[scored.split == "test_seen"], "y", name)
    unseen = metrics.score(scored[scored.split == "test_unseen"], "y", name)
    gap.append({"model": name,
                "rmse seen": seen["rmse"], "rmse unseen": unseen["rmse"],
                "rmse cost %": 100 * (unseen["rmse"] / seen["rmse"] - 1),
                "recall seen": seen["recall"], "recall unseen": unseen["recall"],
                "recall lost %": 100 * (1 - unseen["recall"] / seen["recall"])
                if seen["recall"] else np.nan})
display(pd.DataFrame(gap).round(2))
print("Negative 'lost' means the model transfers better than it performs at home.")

### B3 · Per city
A mean transfer gain hides which cities carry it, and with five unseen cities of very different difficulty that average is the easiest number to mislead with. Cities with no hazardous days show `recall` as NaN rather than 0 — nothing to find is not the same as finding nothing.

In [ ]:
per_city = []
for name in MODELS:
    part = metrics.score_by(scored[scored.split == "test_unseen"], "city", "y", name)
    part.insert(1, "model", name)
    per_city.append(part)
display(pd.concat(per_city).set_index(["city", "model"])
          [["n", "rmse", "support", "recall", "precision", "f1"]]
          .sort_index().round(3))

### B4 · By horizon
Persistence should fall away fastest: today looks less like three days hence than like tomorrow.

In [ ]:
by_h = []
for name in MODELS:
    for horizon, part in scored[scored.split == "test_seen"].groupby("horizon"):
        by_h.append({"model": name, "horizon": int(horizon),
                     **{k: v for k, v in metrics.score(part, "y", name).items()
                        if k in ("rmse", "recall")}})
display(pd.DataFrame(by_h).pivot(index="horizon", columns="model",
                                 values=["rmse", "recall"]).round(3))

## C · Save
The scored forecasts go back to Hugging Face so later notebooks compare against exactly these numbers rather than refitting and getting something slightly different.

In [ ]:
keep = ["city", "date", "target_date", "horizon", "split", "y"] + MODELS
store.upload_df(scored[keep], "processed/baseline_forecasts.parquet",
                "Baseline forecasts: persistence, climatology, gradient boosting")
store.upload_df(table, "processed/baseline_scores.csv", "Baseline scores by split")
print("\u2191 processed/baseline_forecasts.parquet")
print("\u2191 processed/baseline_scores.csv")
print("\nNext: notebook 04 builds the iTransformer backbone and is scored against these.")